In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
CREATE TABLE IF NOT EXISTS ML_BENCHMARK_RESULTS (
    MODEL_CLASS STRING,
    COMPUTE_POOL STRING,
    RUN_ID INT,
    N_COLS_SAMPLED INT,
    N_ROWS_SAMPLED INT,
    DURATION_SECONDS FLOAT,
    START_TIMESTAMP FLOAT
);

In [ ]:
CREATE OR REPLACE STAGE PAYLOAD_STAGE;

In [ ]:
CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_XS_TEST
  AUTO_RESUME = TRUE 
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_XS;
CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_S_TEST
  AUTO_RESUME = TRUE 
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_S;
CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_M_TEST
  AUTO_RESUME = TRUE 
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_M;
CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_SL_TEST
  AUTO_RESUME = TRUE 
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_SL;
-- CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_L_TEST
--   AUTO_RESUME = TRUE 
--   INITALLY_SUSPENDED = TRUE
--   MIN_NODES = 1
--   MAX_NODES = 1
--   INSTANCE_FAMILY = CPU_X64_L;
-- CREATE COMPUTE POOL IF NOT EXISTS HIGHMEM_X64_S_TEST
--   AUTO_RESUME = TRUE 
--   INITALLY_SUSPENDED = TRUE
--   MIN_NODES = 1
--   MAX_NODES = 1
--   INSTANCE_FAMILY = HIGHMEM_X64_S;
-- CREATE COMPUTE POOL IF NOT EXISTS HIGHMEM_X64_M_TEST
--   AUTO_RESUME = TRUE 
--   INITALLY_SUSPENDED = TRUE
--   MIN_NODES = 1
--   MAX_NODES = 1
--   INSTANCE_FAMILY = HIGHMEM_X64_M;
-- CREATE COMPUTE POOL IF NOT EXISTS HIGHMEM_X64_L _TEST
--   AUTO_RESUME = TRUE 
--   INITALLY_SUSPENDED = TRUE
--   MIN_NODES = 1
--   MAX_NODES = 1
--   INSTANCE_FAMILY = HIGHMEM_X64_L;

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
import sys
import pandas as pd

# --- Configuration ---
N_SAMPLES = 1_000_000  # 1 million rows
N_FEATURES = 100       # 100 columns (features)
N_CLASSES = 2          # Binary classification

# --- 1. Data Generation ---
print("Starting data generation...")
# Generate the feature matrix (X) and the target vector (y)
X_full, y_full = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=80,      # 80 useful features (high correlation with target)
    n_redundant=10,        # 10 features that are combinations of informative ones
    n_classes=N_CLASSES,   # Binary classification
    random_state=42,       # For reproducibility
    shuffle=True,
)
print("✅ Data generation complete.")

# --- 2. Memory Analysis ---
# Calculate the memory size of the X matrix in GB
x_memory_gb = X_full.nbytes / (1024**3)

print("\n--- Dataset Summary ---")
print(f"X Shape: {X_full.shape} (Features)")
print(f"y Shape: {y_full.shape} (Labels)")
print(f"X Data Type: {X_full.dtype} ({X_full.dtype.itemsize} bytes per value)")
print(f"Estimated X Memory Footprint: {x_memory_gb:.2f} GB")

In [ ]:
# --- NEW Step 3. Convert and Save to Snowflake Table ---
print("Starting conversion to Snowpark DataFrame and saving to Snowflake...")

# 3a. Combine X and y into a single Pandas DataFrame for easy conversion
# Create feature column names F0, F1, F2, ...
feature_cols = [f'F{i}' for i in range(X_full.shape[1])]
df_combined_pandas = pd.DataFrame(X_full, columns=feature_cols)
df_combined_pandas['TARGET'] = y_full

# 3b. Convert Pandas DataFrame to Snowpark DataFrame
df_snowpark = session.create_dataframe(df_combined_pandas)

# 3c. Save the data to a new Snowflake table
DATA_TABLE_NAME = "BENCHMARK_RAW_DATA"
df_snowpark.write.mode("overwrite").save_as_table(DATA_TABLE_NAME)

print(f"✅ Data saved to Snowflake table: {DATA_TABLE_NAME}")

# --- Set the table name as a global variable needed later ---
# Now the data source is the Snowflake table, not the local NumPy array
# X_full and y_full are no longer needed locally and can be deleted to save memory
del X_full
del y_full

In [ ]:
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier, 
    AdaBoostClassifier,
    VotingClassifier # Useful for combining models later
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Import external libraries
import lightgbm as lgb
import xgboost as xgb
# import torch # Include this if you plan to use PyTorch models

# --- 1. Scikit-learn Ensemble Models (Tree-Based) ---
rf_base_estimator = RandomForestClassifier(
    n_estimators=200,          # Good number of trees
    max_depth=20,              # Moderate depth to prevent deep overfitting
    min_samples_leaf=5,        # Minimum samples per leaf
    max_features='sqrt',       # Recommended for classification
    n_jobs=-1,                 # Use all cores
    random_state=42,           # Reproducibility
)

gb_base_estimator = GradientBoostingClassifier(
    n_estimators=100,          # Number of boosting stages
    learning_rate=0.1,         # Step size shrinkage
    max_depth=3,               # Default is a good starting point
    subsample=0.8,             # Fraction of samples to be used for fitting the individual base learners
    random_state=42,
)

ada_base_estimator = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), # Typically uses a shallow stump
    n_estimators=50,
    learning_rate=1.0,
    random_state=42,
)

# --- 2. Scikit-learn Linear and Kernel Models ---
lr_base_estimator = LogisticRegression(
    penalty='l2',              # Regularization type
    C=1.0,                     # Inverse of regularization strength
    solver='liblinear',        # Good for small datasets
    max_iter=1000,             # Ensure convergence
    random_state=42,
    n_jobs=-1,
)

svm_base_estimator = SVC(
    C=1.0,                     # Regularization parameter
    kernel='rbf',              # Radial Basis Function kernel
    gamma='scale',             # Kernel coefficient auto-adjusted
    probability=True,          # Enable probability estimates (needed for some ensembles)
    random_state=42,
)

# --- 3. Scikit-learn Simple Models ---
knn_base_estimator = KNeighborsClassifier(
    n_neighbors=5,             # Number of neighbors to use
    weights='uniform',         # All points in the neighborhood are weighted equally
    n_jobs=-1,
)

nb_base_estimator = GaussianNB() # Simple and fast, no major hyperparameters

# --- 4. High-Performance Gradient Boosting Frameworks ---
# LightGBM Classifier
lgbm_base_estimator = lgb.LGBMClassifier(
    n_estimators=150,          # Number of boosting stages
    learning_rate=0.05,        # Slower learning rate for better generalization
    num_leaves=31,             # Controls complexity (main parameter)
    max_depth=-1,              # No limit (let num_leaves control complexity)
    n_jobs=-1,
    random_state=42,
    verbose=-1,                # Suppress output
)

# XGBoost Classifier
xgb_base_estimator = xgb.XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=5,               # Control complexity
    subsample=0.8,             # Fraction of samples
    colsample_bytree=0.8,      # Fraction of features
    use_label_encoder=False,   # Recommended setting for newer versions
    eval_metric='logloss',     # Specify evaluation metric
    n_jobs=-1,
    random_state=42,
)

# --- 5. PyTorch Placeholder ---
# Note: A full PyTorch model requires defining an 'nn.Module' class and a training loop.
# This placeholder shows how you might reference it later.
# pytorch_model_placeholder = "Need to define PyTorch_Net class and a wrapper"


base_estimators_lst = [
#    rf_base_estimator,
    gb_base_estimator,
    ada_base_estimator,
    lr_base_estimator,
    svm_base_estimator,
    knn_base_estimator,
    nb_base_estimator,
    lgbm_base_estimator,
    xgb_base_estimator,
    # pytorch_model_placeholder, # Uncomment if you define a wrapper for PyTorch
]

print(f"Total of {len(base_estimators_lst)} estimators configured and ready for training/evaluation.")

In [ ]:
# ============================================================================
# CENTRALIZED CONFIGURATION
# ============================================================================
# All benchmark configuration in one place for easy modification

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  🎯 MAIN CONTROL PARAMETER - Set this to limit total combinations      │
# └─────────────────────────────────────────────────────────────────────────┘
MAX_COMBINATIONS_TO_RUN = 50   # Set to None for unlimited, or a number to cap
                               # Each combination runs RUNS_PER_COMBINATION times
                               # Example: 50 combos × 5 runs = 250 total executions

# --- Table Names ---
DATA_TABLE_NAME = "BENCHMARK_RAW_DATA"
RESULTS_TABLE_NAME = "ML_BENCHMARK_RESULTS"

# --- Data Dimensions ---
NUM_TOTAL_FEATURES = 100

# --- Grid Parameters ---
GRID_COLS = [25, 50, 75, 100]                         # Feature counts to test
GRID_ROWS = [50000, 250000, 450000, 650000, 850000]   # Row counts to test
GRID_POOLS = [
    "CPU_X64_XS_TEST", 
    "CPU_X64_S_TEST", 
    "CPU_X64_M_TEST", 
    "CPU_X64_SL_TEST"
]
RUNS_PER_COMBINATION = 5  # Number of repeated runs per parameter combo

# --- Results Schema (must match table column order for inserts) ---
RESULTS_SCHEMA = [
    'MODEL_CLASS', 
    'COMPUTE_POOL', 
    'RUN_ID', 
    'N_COLS_SAMPLED', 
    'N_ROWS_SAMPLED', 
    'DURATION_SECONDS', 
    'START_TIMESTAMP'
]

# --- Build Estimator Lookup (model name -> instance) ---
ESTIMATOR_LOOKUP = {est.__class__.__name__: est for est in base_estimators_lst}

# --- Calculate grid stats ---
n_estimators = len(ESTIMATOR_LOOKUP)
n_full_grid = n_estimators * len(GRID_COLS) * len(GRID_ROWS) * len(GRID_POOLS)
n_capped = min(n_full_grid, MAX_COMBINATIONS_TO_RUN) if MAX_COMBINATIONS_TO_RUN else n_full_grid

print("✅ Configuration loaded")
print(f"   Data table: {DATA_TABLE_NAME}")
print(f"   Results table: {RESULTS_TABLE_NAME}")
print(f"   Grid: {n_estimators} models × {len(GRID_COLS)} cols × {len(GRID_ROWS)} rows × {len(GRID_POOLS)} pools")
print(f"   Full grid size: {n_full_grid:,} combinations")
print(f"   Estimators: {list(ESTIMATOR_LOOKUP.keys())}")
print(f"\n🎯 MAX_COMBINATIONS_TO_RUN: {MAX_COMBINATIONS_TO_RUN or 'Unlimited'}")
if MAX_COMBINATIONS_TO_RUN and MAX_COMBINATIONS_TO_RUN < n_full_grid:
    print(f"   Will run up to {n_capped} combos × {RUNS_PER_COMBINATION} runs = {n_capped * RUNS_PER_COMBINATION:,} executions")
else:
    print(f"   Will run all {n_full_grid} combos × {RUNS_PER_COMBINATION} runs = {n_full_grid * RUNS_PER_COMBINATION:,} executions")

In [ ]:
# ============================================================================
# LEGACY CONFIG REFERENCE (uses centralized config from Cell 8)
# ============================================================================
# These aliases maintain backwards compatibility with the original job runner

data_cols_lst = GRID_COLS
data_rows_lst = GRID_ROWS
comp_pool_lst = GRID_POOLS
runs_lst = [RUNS_PER_COMBINATION]

print("✅ Grid aliases set from centralized config")


In [ ]:
# ============================================================================
# TEST HARNESS: Validate configuration without running expensive benchmarks
# ============================================================================
import itertools

def run_tests():
    """Run validation tests on the benchmark configuration."""
    results = []
    
    def test(name, condition, details=""):
        status = "✅ PASS" if condition else "❌ FAIL"
        results.append((name, status, details))
        return condition
    
    print("🧪 RUNNING TEST HARNESS")
    print("=" * 70)
    
    all_passed = True
    
    # --- 1. Session & Connection Tests ---
    print("\n📡 SESSION & CONNECTION")
    print("-" * 40)
    
    try:
        session.sql("SELECT 1").collect()
        test("Snowflake session active", True)
    except Exception as e:
        test("Snowflake session active", False, str(e))
        all_passed = False
    
    # --- 2. Table Tests ---
    print("\n📊 TABLE VALIDATION")
    print("-" * 40)
    
    try:
        count = session.table(DATA_TABLE_NAME).count()
        test(f"Data table '{DATA_TABLE_NAME}' exists", True, f"{count:,} rows")
    except Exception as e:
        test(f"Data table '{DATA_TABLE_NAME}' exists", False, str(e))
        all_passed = False
    
    try:
        results_count = session.table(RESULTS_TABLE_NAME).count()
        test(f"Results table '{RESULTS_TABLE_NAME}' exists", True, f"{results_count:,} rows")
    except Exception as e:
        test(f"Results table '{RESULTS_TABLE_NAME}' exists", False, str(e))
        all_passed = False
    
    # --- 3. Configuration Tests ---
    print("\n⚙️ CONFIGURATION VALIDATION")
    print("-" * 40)
    
    test("GRID_COLS defined", len(GRID_COLS) > 0, f"{GRID_COLS}")
    test("GRID_ROWS defined", len(GRID_ROWS) > 0, f"{GRID_ROWS}")
    test("GRID_POOLS defined", len(GRID_POOLS) > 0, f"{len(GRID_POOLS)} pools")
    test("ESTIMATOR_LOOKUP populated", len(ESTIMATOR_LOOKUP) > 0, f"{len(ESTIMATOR_LOOKUP)} estimators")
    test("RUNS_PER_COMBINATION > 0", RUNS_PER_COMBINATION > 0, f"{RUNS_PER_COMBINATION}")
    test("RESULTS_SCHEMA has 7 columns", len(RESULTS_SCHEMA) == 7, f"{len(RESULTS_SCHEMA)} columns")
    
    max_cap = MAX_COMBINATIONS_TO_RUN
    test("MAX_COMBINATIONS_TO_RUN valid", max_cap is None or max_cap > 0, 
         f"{'Unlimited' if max_cap is None else max_cap}")
    
    # --- 4. Estimator Tests ---
    print("\n🤖 ESTIMATOR VALIDATION")
    print("-" * 40)
    
    for name, est in ESTIMATOR_LOOKUP.items():
        has_fit = hasattr(est, 'fit')
        has_params = hasattr(est, 'get_params')
        passed = has_fit and has_params
        test(f"Estimator '{name}'", passed, 
             f"fit={'✓' if has_fit else '✗'}, get_params={'✓' if has_params else '✗'}")
        if not passed:
            all_passed = False
    
    # --- 5. Grid Math Tests ---
    print("\n🔢 GRID CALCULATIONS")
    print("-" * 40)
    
    n_models = len(ESTIMATOR_LOOKUP)
    n_cols = len(GRID_COLS)
    n_rows = len(GRID_ROWS)
    n_pools = len(GRID_POOLS)
    full_grid = n_models * n_cols * n_rows * n_pools
    
    test("Grid multiplication correct", full_grid == n_full_grid, 
         f"{n_models}×{n_cols}×{n_rows}×{n_pools} = {full_grid}")
    
    effective_combos = min(full_grid, MAX_COMBINATIONS_TO_RUN) if MAX_COMBINATIONS_TO_RUN else full_grid
    total_runs = effective_combos * RUNS_PER_COMBINATION
    
    print(f"\n   Full grid:        {full_grid:,} combinations")
    print(f"   After cap:        {effective_combos:,} combinations")
    print(f"   Total executions: {total_runs:,} (× {RUNS_PER_COMBINATION} runs each)")
    
    # --- 6. Sample Combination Test ---
    print("\n🧩 SAMPLE COMBINATION")
    print("-" * 40)
    
    sample_model = list(ESTIMATOR_LOOKUP.keys())[0]
    sample_pool = GRID_POOLS[0]
    sample_cols = GRID_COLS[0]
    sample_rows = GRID_ROWS[0]
    
    print(f"   Model:   {sample_model}")
    print(f"   Pool:    {sample_pool}")
    print(f"   Columns: {sample_cols}")
    print(f"   Rows:    {sample_rows:,}")
    
    sample_est = ESTIMATOR_LOOKUP[sample_model]
    test("Sample estimator instantiable", True, f"{sample_est.__class__.__name__}")
    test("Sample params extractable", True, f"{len(sample_est.get_params())} params")
    
    # --- 7. Schema Test ---
    print("\n📋 SCHEMA VALIDATION")
    print("-" * 40)
    
    expected_schema = ['MODEL_CLASS', 'COMPUTE_POOL', 'RUN_ID', 'N_COLS_SAMPLED', 
                       'N_ROWS_SAMPLED', 'DURATION_SECONDS', 'START_TIMESTAMP']
    schema_match = RESULTS_SCHEMA == expected_schema
    test("Schema matches expected", schema_match, 
         "OK" if schema_match else f"Got: {RESULTS_SCHEMA}")
    if not schema_match:
        all_passed = False
    
    # --- Summary ---
    print("\n" + "=" * 70)
    n_pass = sum(1 for _, s, _ in results if "PASS" in s)
    n_fail = sum(1 for _, s, _ in results if "FAIL" in s)
    
    if all_passed:
        print(f"🎉 ALL TESTS PASSED ({n_pass}/{n_pass + n_fail})")
        print("   Configuration is valid. Ready to run benchmarks!")
    else:
        print(f"⚠️ SOME TESTS FAILED ({n_fail} failures)")
        print("   Review failures above before running benchmarks.")
        for name, status, details in results:
            if "FAIL" in status:
                print(f"   ❌ {name}: {details}")
    
    return all_passed

# Run the tests
test_passed = run_tests()

In [ ]:
# ============================================================================
# COVERAGE ANALYSIS: Query existing results and identify tested combinations
# ============================================================================
import itertools
import pandas as pd

print("🔍 BENCHMARK COVERAGE ANALYSIS")
print("=" * 70)

# --- 1. Load Results from Snowflake ---
results_all_df = session.table(RESULTS_TABLE_NAME).to_pandas()
print(f"📊 Total benchmark records: {len(results_all_df):,}")

# Separate successful vs failed runs
results_success_df = results_all_df[results_all_df['DURATION_SECONDS'] > 0].copy()
n_success = len(results_success_df)
n_failed = len(results_all_df) - n_success
print(f"✅ Successful runs: {n_success:,}")
print(f"❌ Failed runs: {n_failed:,}")

# --- 2. Extract Unique Tested Combinations ---
# A combination is (MODEL_CLASS, COMPUTE_POOL, N_COLS_SAMPLED, N_ROWS_SAMPLED)
combo_cols = ['MODEL_CLASS', 'COMPUTE_POOL', 'N_COLS_SAMPLED', 'N_ROWS_SAMPLED']
tested_combos_df = results_success_df.groupby(combo_cols).size().reset_index(name='run_count')
tested_combos_set = set(tested_combos_df[combo_cols].apply(tuple, axis=1))
print(f"🔑 Unique tested combinations: {len(tested_combos_set):,}")

# --- 3. Coverage by Model Type ---
print(f"\n📈 COVERAGE BY MODEL TYPE")
print("-" * 50)
model_stats = results_success_df.groupby('MODEL_CLASS').agg({
    'DURATION_SECONDS': ['count', 'mean', 'std', 'min', 'max']
}).round(2)
model_stats.columns = ['runs', 'mean_s', 'std_s', 'min_s', 'max_s']
print(model_stats.sort_values('runs', ascending=False))

# --- 4. Coverage by Compute Pool ---
print(f"\n🖥️ COVERAGE BY COMPUTE POOL")
print("-" * 50)
pool_stats = results_success_df.groupby('COMPUTE_POOL').agg({
    'DURATION_SECONDS': ['count', 'mean', 'std']
}).round(2)
pool_stats.columns = ['runs', 'mean_s', 'std_s']
print(pool_stats.sort_values('runs', ascending=False))

# --- 5. Coverage Matrix: Rows × Columns ---
print(f"\n📊 COVERAGE MATRIX (Rows × Cols) - Run counts")
print("-" * 60)
rows_cols_matrix = results_success_df.pivot_table(
    values='DURATION_SECONDS', 
    index='N_ROWS_SAMPLED', 
    columns='N_COLS_SAMPLED',
    aggfunc='count',
    fill_value=0
)
print(rows_cols_matrix)

# --- 6. Coverage Matrix: Model × Pool ---
print(f"\n🗺️ COVERAGE MATRIX (Model × Pool) - Run counts")
print("-" * 60)
model_pool_matrix = results_success_df.pivot_table(
    values='DURATION_SECONDS',
    index='MODEL_CLASS',
    columns='COMPUTE_POOL', 
    aggfunc='count',
    fill_value=0
)
print(model_pool_matrix)

# --- 7. Discover Unique Values Present in Data ---
print(f"\n📋 DISCOVERED VALUES IN EXISTING DATA")
print("-" * 50)
discovered_models = sorted(results_all_df['MODEL_CLASS'].unique().tolist())
discovered_pools = sorted(results_all_df['COMPUTE_POOL'].unique().tolist())
discovered_cols = sorted(results_all_df['N_COLS_SAMPLED'].unique().tolist())
discovered_rows = sorted(results_all_df['N_ROWS_SAMPLED'].unique().tolist())

print(f"Models ({len(discovered_models)}): {discovered_models}")
print(f"Pools ({len(discovered_pools)}): {discovered_pools}")  
print(f"Columns ({len(discovered_cols)}): {discovered_cols}")
print(f"Rows ({len(discovered_rows)}): {discovered_rows}")

# --- Store for downstream cells ---
# Using clear, descriptive names
coverage_tested_combos = tested_combos_set
coverage_discovered = {
    'models': discovered_models,
    'pools': discovered_pools,
    'cols': discovered_cols,
    'rows': discovered_rows
}

print(f"\n✅ Coverage analysis complete.")
print(f"   Stored: coverage_tested_combos ({len(coverage_tested_combos)} combos)")
print(f"   Stored: coverage_discovered (dict with models/pools/cols/rows)")

In [ ]:
# ============================================================================
# NEGATIVE SPACE ANALYSIS: Identify untested benchmark combinations
# ============================================================================
import itertools

print("🔲 NEGATIVE SPACE ANALYSIS - Finding Untested Combinations")
print("=" * 70)

# --- Grid Source Selection ---
# True  = Fill gaps in existing data (use discovered values)
# False = Use custom grid (can expand beyond current data)
USE_DISCOVERED_GRID = True

if USE_DISCOVERED_GRID:
    # Use values discovered from existing benchmark results
    target_grid = {
        'models': coverage_discovered['models'],
        'pools': coverage_discovered['pools'],
        'cols': coverage_discovered['cols'],
        'rows': coverage_discovered['rows']
    }
    print("📊 Using DISCOVERED grid (fill gaps in existing data)")
else:
    # Define custom grid - modify these to expand testing scope
    target_grid = {
        'models': [
            'AdaBoostClassifier', 'GaussianNB', 'GradientBoostingClassifier',
            'KNeighborsClassifier', 'LGBMClassifier', 'LogisticRegression',
            'SVC', 'XGBClassifier',
            # 'RandomForestClassifier',  # Uncomment to add
        ],
        'pools': GRID_POOLS,  # From centralized config
        'cols': GRID_COLS,    # From centralized config  
        'rows': GRID_ROWS,    # From centralized config
    }
    print("📊 Using CUSTOM grid definition")

# --- Calculate Full Grid vs Tested ---
full_grid_set = set(itertools.product(
    target_grid['models'], 
    target_grid['pools'], 
    target_grid['cols'], 
    target_grid['rows']
))
untested_combos = full_grid_set - coverage_tested_combos
n_tested = len(coverage_tested_combos)
n_untested = len(untested_combos)
n_total = len(full_grid_set)

print(f"\n📐 GRID DIMENSIONS:")
print(f"   Models:  {len(target_grid['models'])}")
print(f"   Pools:   {len(target_grid['pools'])}")
print(f"   Columns: {len(target_grid['cols'])}")
print(f"   Rows:    {len(target_grid['rows'])}")

print(f"\n📊 COVERAGE SUMMARY:")
print(f"   Full grid size:    {n_total:,} combinations")
print(f"   Already tested:    {n_tested:,} combinations")
print(f"   Untested (gaps):   {n_untested:,} combinations")
coverage_pct = (n_tested / n_total * 100) if n_total > 0 else 0
print(f"   Coverage:          {coverage_pct:.1f}%")

# --- Analyze Untested Combinations ---
if n_untested > 0:
    untested_df = pd.DataFrame(
        list(untested_combos), 
        columns=['MODEL_CLASS', 'COMPUTE_POOL', 'N_COLS_SAMPLED', 'N_ROWS_SAMPLED']
    )
    
    print(f"\n🔍 UNTESTED COMBINATIONS BY DIMENSION:")
    print("-" * 50)
    
    print(f"\nBy Model Type:")
    print(untested_df['MODEL_CLASS'].value_counts().to_string())
    
    print(f"\nBy Compute Pool:")
    print(untested_df['COMPUTE_POOL'].value_counts().to_string())
    
    print(f"\nBy Column Count:")
    print(untested_df['N_COLS_SAMPLED'].value_counts().sort_index().to_string())
    
    print(f"\nBy Row Count:")
    print(untested_df['N_ROWS_SAMPLED'].value_counts().sort_index().to_string())
    
    # Untested combinations matrix
    print(f"\n📊 UNTESTED MATRIX (Model × Pool):")
    print("-" * 50)
    untested_matrix = untested_df.pivot_table(
        index='MODEL_CLASS',
        columns='COMPUTE_POOL',
        aggfunc='size',
        fill_value=0
    )
    print(untested_matrix)
    
    # Preview
    print(f"\n👀 PREVIEW: First 20 untested combinations:")
    print("-" * 70)
    sorted_untested = sorted(untested_combos)
    for i, (model, pool, cols, rows) in enumerate(sorted_untested[:20]):
        print(f"  {i+1:2d}. {model:30s} | {pool:18s} | cols={cols:3d} | rows={rows:,}")
    if n_untested > 20:
        print(f"  ... and {n_untested - 20} more")
else:
    print(f"\n✅ COMPLETE COVERAGE! All combinations have been tested.")

# --- Store for downstream cells ---
gaps_untested_combos = untested_combos
gaps_target_grid = target_grid

print(f"\n✅ Analysis complete.")
print(f"   Stored: gaps_untested_combos ({len(gaps_untested_combos)} combos)")
print(f"   Stored: gaps_target_grid (dict with target parameters)")

In [ ]:
# ============================================================================
# RUN UNTESTED BENCHMARKS: Fill gaps in the benchmark grid
# ============================================================================
import time
import numpy as np
import concurrent.futures
from snowflake.ml.jobs import remote
from snowflake.snowpark import Session

print("🚀 FILLING GAPS - Running Untested Benchmark Combinations")
print("=" * 70)

# Uses configuration from Cell 8 and gaps from Cell 11
print(f"📦 Available estimators: {list(ESTIMATOR_LOOKUP.keys())}")

# --- Check if there are gaps to fill ---
if len(gaps_untested_combos) == 0:
    print("\n✅ No untested combinations! Grid is complete.")
else:
    n_gaps = len(gaps_untested_combos)
    print(f"\n📋 {n_gaps:,} untested combinations in grid")
    
    # --- Filter to runnable combos (estimator must exist) ---
    runnable_combos = [
        combo for combo in gaps_untested_combos 
        if combo[0] in ESTIMATOR_LOOKUP
    ]
    n_skipped_missing = n_gaps - len(runnable_combos)
    if n_skipped_missing > 0:
        print(f"⚠️  Skipping {n_skipped_missing} combos (estimator not available)")
    
    # --- Apply MAX_COMBINATIONS_TO_RUN cap ---
    if MAX_COMBINATIONS_TO_RUN and len(runnable_combos) > MAX_COMBINATIONS_TO_RUN:
        print(f"🎯 Capping to MAX_COMBINATIONS_TO_RUN = {MAX_COMBINATIONS_TO_RUN}")
        runnable_combos = runnable_combos[:MAX_COMBINATIONS_TO_RUN]
    
    print(f"\n📋 Will run {len(runnable_combos):,} combinations")
    print(f"   × {RUNS_PER_COMBINATION} runs each = {len(runnable_combos) * RUNS_PER_COMBINATION:,} total executions")
    
    # --- Group by compute pool for parallel execution ---
    combos_by_pool = {}
    for model_name, pool_name, n_cols, n_rows in runnable_combos:
        combos_by_pool.setdefault(pool_name, []).append((model_name, pool_name, n_cols, n_rows))
    
    print(f"\n📊 Jobs per compute pool:")
    for pool, combos in sorted(combos_by_pool.items()):
        print(f"   {pool}: {len(combos)} combinations")
    
    job_counter = [0]  # Use list for mutable counter in nested function
    n_total_jobs = len(runnable_combos)
    
    def run_pool_jobs_sequential(pool_name, pool_combos):
        """Run all jobs for a pool sequentially (one at a time)."""
        n_completed = 0
        n_failed = 0
        
        print(f"\n🚀 Starting jobs for pool: {pool_name}")
        
        for model_name, pool, n_cols, n_rows in pool_combos:
            estimator = ESTIMATOR_LOOKUP[model_name]
            estimator_class = estimator.__class__
            estimator_params = estimator.get_params()
            
            # Define remote job function
            @remote(pool, stage_name="PAYLOAD_STAGE", session=session)
            def benchmark_job(model_class_name, pool_name, est_class, est_params, 
                            n_cols, n_rows, n_runs, data_table, n_features, results_table):
                """Remote job: train model and log timing results."""
                sess = Session.get_active_session()
                
                # Load data
                raw_df = sess.table(data_table)
                feature_cols = [c for c in raw_df.columns if c != 'TARGET']
                pdf = raw_df.to_pandas()
                X_all = pdf[feature_cols].to_numpy()
                y_all = pdf['TARGET'].to_numpy()
                
                # Run multiple iterations
                for run_id in range(1, n_runs + 1):
                    # Sample rows and columns
                    row_idx = np.random.choice(X_all.shape[0], size=n_rows, replace=False)
                    col_idx = np.random.choice(n_features, size=n_cols, replace=False)
                    X = X_all[row_idx][:, col_idx]
                    y = y_all[row_idx]
                    
                    # Train and time
                    model = est_class(**est_params)
                    start_ts = time.time()
                    try:
                        model.fit(X, y)
                        duration = time.time() - start_ts
                    except Exception as e:
                        duration = -1  # Mark as failed
                        print(f"❌ Run {run_id} failed: {e}")
                    
                    # Log result
                    result = [(model_class_name, pool_name, run_id, n_cols, n_rows, duration, start_ts)]
                    result_df = sess.create_dataframe(result, schema=RESULTS_SCHEMA)
                    result_df.write.mode("append").save_as_table(results_table)
                    print(f"✅ {model_class_name} run {run_id}/{n_runs}: {duration:.3f}s")
                
                return f"Completed {n_runs} runs for {model_class_name}"
            
            # Submit and wait
            job_counter[0] += 1
            print(f"[{job_counter[0]}/{n_total_jobs}] {model_name} | {pool} | {n_cols} cols | {n_rows:,} rows")
            
            job = benchmark_job(
                model_name, pool, estimator_class, estimator_params,
                n_cols, n_rows, RUNS_PER_COMBINATION,
                DATA_TABLE_NAME, NUM_TOTAL_FEATURES, RESULTS_TABLE_NAME
            )
            
            try:
                job.wait()
                n_completed += 1
            except Exception as e:
                n_failed += 1
                print(f"❌ Job failed: {e}")
        
        return pool_name, n_completed, n_failed

    # --- Execute pools in parallel ---
    pool_results = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(combos_by_pool)) as executor:
        futures = {
            executor.submit(run_pool_jobs_sequential, pool, combos): pool
            for pool, combos in combos_by_pool.items()
        }
        
        for future in concurrent.futures.as_completed(futures):
            pool = futures[future]
            try:
                _, completed, failed = future.result()
                pool_results[pool] = {'completed': completed, 'failed': failed}
            except Exception as e:
                print(f"💥 Pool {pool} failed entirely: {e}")
                pool_results[pool] = {'completed': 0, 'failed': -1}

    # --- Summary ---
    print("\n" + "=" * 60)
    print("BENCHMARK GAP-FILL SUMMARY")
    print("=" * 60)
    
    total_ok = sum(r['completed'] for r in pool_results.values() if r['failed'] != -1)
    total_fail = sum(r['failed'] for r in pool_results.values() if r['failed'] != -1)
    
    for pool, result in sorted(pool_results.items()):
        if result['failed'] == -1:
            print(f"💥 {pool:20} | POOL FAILED")
        else:
            rate = result['completed'] / max(1, result['completed'] + result['failed']) * 100
            print(f"   {pool:20} | ✅ {result['completed']:3d} | ❌ {result['failed']:3d} | {rate:.0f}%")
    
    print("-" * 60)
    print(f"Total: {total_ok} successful, {total_fail} failed")
    print(f"Results saved to: {RESULTS_TABLE_NAME}")

In [ ]:
# ============================================================================
# FULL GRID BENCHMARK: Run all combinations (initial population)
# ============================================================================
# NOTE: Use Cell 12 (gap-fill) instead if you only want to run missing combos
# This cell runs the FULL grid from scratch

import itertools
import time
import numpy as np
import concurrent.futures
from snowflake.ml.jobs import remote
from snowflake.snowpark import Session

print("🚀 FULL GRID BENCHMARK - Running All Combinations")
print("=" * 70)
print("⚠️  This runs the ENTIRE grid. Use Cell 13 (gap-fill) to only fill gaps.")

# --- Build full grid from centralized config ---
full_combinations = list(itertools.product(
    base_estimators_lst,  # Estimator instances
    GRID_COLS,            # From Cell 8 config
    GRID_ROWS,            # From Cell 8 config  
    GRID_POOLS,           # From Cell 8 config
))

print(f"\n📋 Full grid: {len(full_combinations):,} combinations")

# --- Apply MAX_COMBINATIONS_TO_RUN cap ---
if MAX_COMBINATIONS_TO_RUN and len(full_combinations) > MAX_COMBINATIONS_TO_RUN:
    print(f"🎯 Capping to MAX_COMBINATIONS_TO_RUN = {MAX_COMBINATIONS_TO_RUN}")
    full_combinations = full_combinations[:MAX_COMBINATIONS_TO_RUN]

# Group by compute pool
combos_by_pool = {}
for estimator, n_cols, n_rows, pool in full_combinations:
    combos_by_pool.setdefault(pool, []).append((estimator, n_cols, n_rows, pool))

n_total_jobs = len(full_combinations)
print(f"\n📋 Will run: {n_total_jobs:,} combinations")
print(f"   × {RUNS_PER_COMBINATION} runs each = {n_total_jobs * RUNS_PER_COMBINATION:,} total executions")
print(f"\n📊 Jobs per compute pool:")
for pool, combos in sorted(combos_by_pool.items()):
    print(f"   {pool}: {len(combos)} combinations")

job_counter = [0]  # Mutable counter for nested function

def run_pool_full_grid(pool_name, pool_combos):
    """Run all grid combinations for a single pool (sequential within pool)."""
    n_completed = 0
    n_failed = 0
    
    print(f"\n🚀 Starting pool: {pool_name}")
    
    for estimator, n_cols, n_rows, pool in pool_combos:
        model_name = estimator.__class__.__name__
        estimator_class = estimator.__class__
        estimator_params = estimator.get_params()
        
        @remote(pool, stage_name="PAYLOAD_STAGE", session=session)
        def benchmark_job(model_class_name, pool_name, est_class, est_params,
                         n_cols, n_rows, n_runs, data_table, n_features, results_table):
            """Remote benchmark job."""
            sess = Session.get_active_session()
            
            # Load data
            raw_df = sess.table(data_table)
            feature_cols = [c for c in raw_df.columns if c != 'TARGET']
            pdf = raw_df.to_pandas()
            X_all = pdf[feature_cols].to_numpy()
            y_all = pdf['TARGET'].to_numpy()
            
            for run_id in range(1, n_runs + 1):
                # Sample data
                row_idx = np.random.choice(X_all.shape[0], size=n_rows, replace=False)
                col_idx = np.random.choice(n_features, size=n_cols, replace=False)
                X = X_all[row_idx][:, col_idx]
                y = y_all[row_idx]
                
                # Train and time
                model = est_class(**est_params)
                start_ts = time.time()
                try:
                    model.fit(X, y)
                    duration = time.time() - start_ts
                except Exception as e:
                    duration = -1
                    print(f"❌ Run {run_id} failed: {e}")
                
                # Log result
                result = [(model_class_name, pool_name, run_id, n_cols, n_rows, duration, start_ts)]
                result_df = sess.create_dataframe(result, schema=RESULTS_SCHEMA)
                result_df.write.mode("append").save_as_table(results_table)
                print(f"✅ {model_class_name} run {run_id}/{n_runs}: {duration:.3f}s")
            
            return f"Done: {model_class_name}"
        
        job_counter[0] += 1
        print(f"[{job_counter[0]}/{n_total_jobs}] {model_name} | {pool} | {n_cols} cols | {n_rows:,} rows")
        
        job = benchmark_job(
            model_name, pool, estimator_class, estimator_params,
            n_cols, n_rows, RUNS_PER_COMBINATION,
            DATA_TABLE_NAME, NUM_TOTAL_FEATURES, RESULTS_TABLE_NAME
        )
        
        try:
            job.wait()
            n_completed += 1
        except Exception as e:
            n_failed += 1
            print(f"❌ Job failed: {e}")
    
    return pool_name, n_completed, n_failed

# --- Execute pools in parallel ---
pool_results = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=len(combos_by_pool)) as executor:
    futures = {
        executor.submit(run_pool_full_grid, pool, combos): pool
        for pool, combos in combos_by_pool.items()
    }
    
    for future in concurrent.futures.as_completed(futures):
        pool = futures[future]
        try:
            _, completed, failed = future.result()
            pool_results[pool] = {'completed': completed, 'failed': failed}
        except Exception as e:
            print(f"💥 Pool {pool} failed: {e}")
            pool_results[pool] = {'completed': 0, 'failed': -1}

# --- Summary ---
print("\n" + "=" * 60)
print("FULL GRID BENCHMARK SUMMARY")
print("=" * 60)

total_ok = sum(r['completed'] for r in pool_results.values() if r['failed'] != -1)
total_fail = sum(r['failed'] for r in pool_results.values() if r['failed'] != -1)

for pool, result in sorted(pool_results.items()):
    if result['failed'] == -1:
        print(f"💥 {pool:20} | POOL FAILED")
    else:
        rate = result['completed'] / max(1, result['completed'] + result['failed']) * 100
        print(f"   {pool:20} | ✅ {result['completed']:3d} | ❌ {result['failed']:3d} | {rate:.0f}%")

print("-" * 60)
print(f"Total: {total_ok} successful, {total_fail} failed")
print(f"Results saved to: {RESULTS_TABLE_NAME}")

In [ ]:
# Comprehensive statistical summary of ML_BENCHMARK_RESULTS
import pandas as pd
from scipy import stats
import numpy as np

print("📊 COMPREHENSIVE STATISTICAL SUMMARY - ML_BENCHMARK_RESULTS")
print("=" * 70)

# Load data
df = session.table("ML_BENCHMARK_RESULTS").to_pandas()
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Basic info
print(f"\n🔍 DATA OVERVIEW")
print("-" * 30)
print(df.info())

# Missing values
print(f"\n❌ MISSING VALUES")
print("-" * 30)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)
print(missing_summary[missing_summary['Missing Count'] > 0])
if missing_summary['Missing Count'].sum() == 0:
    print("✅ No missing values found")

# Numerical columns analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print(f"\n📈 NUMERICAL VARIABLES SUMMARY")
    print("-" * 40)
    
    # Enhanced describe
    desc = df[numeric_cols].describe()
    
    # Add additional statistics
    for col in numeric_cols:
        if len(df[col].dropna()) > 0:
            desc.loc['mode', col] = df[col].mode().iloc[0] if not df[col].mode().empty else np.nan
            desc.loc['skewness', col] = stats.skew(df[col].dropna())
            desc.loc['kurtosis', col] = stats.kurtosis(df[col].dropna())
            desc.loc['range', col] = df[col].max() - df[col].min()
            desc.loc['iqr', col] = df[col].quantile(0.75) - df[col].quantile(0.25)
    
    print(desc.round(3))

# Failed runs analysis
if 'DURATION_SECONDS' in df.columns:
    print(f"\n⚠️  DURATION ANALYSIS")
    print("-" * 30)
    failed_runs = (df['DURATION_SECONDS'] <= 0).sum()
    success_runs = (df['DURATION_SECONDS'] > 0).sum()
    print(f"Successful runs: {success_runs:,} ({success_runs/len(df)*100:.1f}%)")
    print(f"Failed runs: {failed_runs:,} ({failed_runs/len(df)*100:.1f}%)")
    
    if success_runs > 0:
        valid_durations = df[df['DURATION_SECONDS'] > 0]['DURATION_SECONDS']
        print(f"Duration stats (successful runs only):")
        print(f"  Mean: {valid_durations.mean():.3f}s")
        print(f"  Median: {valid_durations.median():.3f}s") 
        print(f"  Min: {valid_durations.min():.3f}s")
        print(f"  Max: {valid_durations.max():.3f}s")
        print(f"  Std Dev: {valid_durations.std():.3f}s")

# Categorical columns analysis
categorical_cols = df.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    print(f"\n📝 CATEGORICAL VARIABLES SUMMARY")
    print("-" * 40)
    
    for col in categorical_cols:
        print(f"\n{col}:")
        value_counts = df[col].value_counts()
        print(f"  Unique values: {df[col].nunique()}")
        print(f"  Most frequent: '{value_counts.index[0]}' ({value_counts.iloc[0]:,} times, {value_counts.iloc[0]/len(df)*100:.1f}%)")
        if len(value_counts) > 1:
            print(f"  Least frequent: '{value_counts.index[-1]}' ({value_counts.iloc[-1]:,} times, {value_counts.iloc[-1]/len(df)*100:.1f}%)")
        
        print("  Top 5 values:")
        for idx, (val, count) in enumerate(value_counts.head().items()):
            print(f"    {idx+1}. '{val}': {count:,} ({count/len(df)*100:.1f}%)")

# Correlation analysis for numeric columns
if len(numeric_cols) > 1:
    print(f"\n🔗 CORRELATION MATRIX")
    print("-" * 30)
    corr_matrix = df[numeric_cols].corr()
    print(corr_matrix.round(3))
    
    # Find highest correlations
    print(f"\nHighest correlations:")
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    high_corr = corr_matrix.mask(mask).stack().sort_values(key=abs, ascending=False).head(3)
    for (var1, var2), corr_val in high_corr.items():
        print(f"  {var1} ↔ {var2}: {corr_val:.3f}")

# Distribution analysis
if 'DURATION_SECONDS' in df.columns:
    valid_durations = df[df['DURATION_SECONDS'] > 0]['DURATION_SECONDS']
    if len(valid_durations) > 0:
        print(f"\n📊 DURATION DISTRIBUTION (Successful runs)")
        print("-" * 45)
        
        # Percentiles
        percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
        print("Percentiles:")
        for p in percentiles:
            val = np.percentile(valid_durations, p)
            print(f"  P{p:2d}: {val:8.3f}s")
        
        # Quick histogram
        print(f"\nDuration ranges (successful runs):")
        bins = [0, 1, 5, 10, 30, 60, 300, float('inf')]
        labels = ['<1s', '1-5s', '5-10s', '10-30s', '30-60s', '1-5m', '>5m']
        duration_ranges = pd.cut(valid_durations, bins=bins, labels=labels, right=False)
        range_counts = duration_ranges.value_counts().sort_index()
        for label, count in range_counts.items():
            pct = count / len(valid_durations) * 100
            print(f"  {label:>5}: {count:6,} ({pct:5.1f}%)")

print(f"\n✅ Summary complete!")

In [ ]:
# Two-tiered ML model with fallback for datasets with no failures
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score
import numpy as np

print("🎯 Training Two-Tiered ML Model...")
print("=" * 50)

# Load and prep data
results_df = session.table("ML_BENCHMARK_RESULTS").to_pandas()
print(f"Total runs: {len(results_df):,}")

# Check success/failure distribution
success_count = (results_df['DURATION_SECONDS'] > 0).sum()
failure_count = (results_df['DURATION_SECONDS'] <= 0).sum()
print(f"Success runs: {success_count:,}, Failed runs: {failure_count:,}")

# Encode categorical variables
le_model = LabelEncoder()
le_pool = LabelEncoder()
results_df['MODEL_ENCODED'] = le_model.fit_transform(results_df['MODEL_CLASS'])
results_df['POOL_ENCODED'] = le_pool.fit_transform(results_df['COMPUTE_POOL'])

feature_cols = ['MODEL_ENCODED', 'POOL_ENCODED', 'N_COLS_SAMPLED', 'N_ROWS_SAMPLED']
X = results_df[feature_cols]

# Initialize variables
classifier = None
class_accuracy = None

# --- TIER 1: SUCCESS/FAILURE CLASSIFICATION (if needed) ---
if failure_count > 0:
    print("\n🔍 TIER 1: Training Success/Failure Classifier")
    print("-" * 45)
    
    y_success = (results_df['DURATION_SECONDS'] > 0).astype(int)
    success_rate = y_success.mean()
    print(f"Overall success rate: {success_rate:.1%}")
    
    try:
        # Train classification model
        X_train, X_test, y_train_class, y_test_class = train_test_split(
            X, y_success, test_size=0.2, random_state=42, stratify=y_success
        )
        
        classifier = LogisticRegression(random_state=42, max_iter=1000)
        classifier.fit(X_train, y_train_class)
        
        # Evaluate classifier
        y_pred_class = classifier.predict(X_test)
        class_accuracy = accuracy_score(y_test_class, y_pred_class)
        print(f"Classification Accuracy: {class_accuracy:.3f}")
        
    except ValueError as e:
        print(f"⚠️ Classification model failed: {e}")
        print("📝 Assuming all runs will succeed (no failure pattern detected)")
        classifier = None
else:
    print("\n✅ TIER 1: Skipped - No failed runs detected")
    print("📝 All runs successful, assuming 100% success rate")

# --- TIER 2: DURATION REGRESSION ---
print("\n⏱️ TIER 2: Training Duration Regressor")
print("-" * 40)

# Filter to successful runs only for training
success_df = results_df[results_df['DURATION_SECONDS'] > 0].copy()
X_success = success_df[feature_cols]
y_duration = success_df['DURATION_SECONDS']

print(f"Training on {len(success_df):,} successful runs")

# Train regression model
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_success, y_duration, test_size=0.2, random_state=42
)

regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
regressor.fit(X_train_reg, y_train_reg)

# Evaluate regressor
y_pred_reg = regressor.predict(X_test_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
mae = mean_absolute_error(y_test_reg, y_pred_reg)

print(f"Regression R²: {r2:.3f}")
print(f"Regression MAE: {mae:.2f}s")

# --- COMBINED PREDICTION FUNCTION ---
def predict_ml_execution(model_class, compute_pool, n_cols, n_rows, prob_threshold=0.5):
    """
    Two-tiered prediction: First predicts if run will succeed, then predicts duration
    """
    try:
        model_enc = le_model.transform([model_class])[0]
        pool_enc = le_pool.transform([compute_pool])[0]
    except ValueError:
        return {
            'will_succeed': False,
            'success_probability': 0.0,
            'predicted_duration': None,
            'confidence': 'Unknown model/pool combination'
        }
    
    features = np.array([[model_enc, pool_enc, n_cols, n_rows]])
    
    # Tier 1: Predict success probability (if classifier exists)
    if classifier is not None:
        success_prob = classifier.predict_proba(features)[0, 1]
        will_succeed = success_prob >= prob_threshold
    else:
        # No failures in training data, assume success
        success_prob = 1.0
        will_succeed = True
    
    # Tier 2: Predict duration if likely to succeed
    predicted_duration = None
    confidence = "Low"
    
    if will_succeed:
        predicted_duration = regressor.predict(features)[0]
        
        # Confidence based on probability
        if success_prob >= 0.9:
            confidence = "High"
        elif success_prob >= 0.7:
            confidence = "Medium"
        else:
            confidence = "Low"
    
    return {
        'will_succeed': will_succeed,
        'success_probability': success_prob,
        'predicted_duration': predicted_duration,
        'confidence': confidence
    }

# --- TESTING THE COMBINED MODEL ---
print(f"\n🧪 TESTING COMBINED MODEL")
print("-" * 30)

test_cases = [
    ('LogisticRegression', 'CPU_X64_XS_TEST', 25, 50000),
    ('RandomForestClassifier', 'CPU_X64_S_TEST', 50, 250000),
    ('XGBClassifier', 'CPU_X64_M_TEST', 75, 650000),
    ('SVC', 'CPU_X64_SL_TEST', 100, 1000000),
    ('KNeighborsClassifier', 'CPU_X64_XS_TEST', 100, 1000000)
]

for model_class, compute_pool, n_cols, n_rows in test_cases:
    result = predict_ml_execution(model_class, compute_pool, n_cols, n_rows)
    
    print(f"\n{model_class} on {compute_pool}")
    print(f"  Data: {n_cols} cols, {n_rows:,} rows")
    print(f"  Will succeed: {result['will_succeed']} ({result['success_probability']:.1%} confidence)")
    if result['predicted_duration']:
        print(f"  Predicted duration: {result['predicted_duration']:.1f}s")
    print(f"  Confidence: {result['confidence']}")

print(f"\n✅ Two-Tiered Model Complete!")
if class_accuracy:
    print(f"📈 Tier 1 (Success): {class_accuracy:.1%} accuracy")
else:
    print(f"📈 Tier 1 (Success): Skipped - no failures detected")
print(f"📊 Tier 2 (Duration): R²={r2:.3f}, MAE={mae:.1f}s")
print(f"🎯 Use predict_ml_execution() for combined predictions")

In [ ]:
# ============================================================================
# VISUALIZATIONS: Benchmark Results Analysis
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("📊 Generating Benchmark Visualizations...")

# --- Load fresh data ---
viz_df = session.table(RESULTS_TABLE_NAME).to_pandas()
viz_df = viz_df[viz_df['DURATION_SECONDS'] > 0].copy()  # Only successful runs

if len(viz_df) == 0:
    print("⚠️ No successful benchmark data to visualize!")
else:
    # Set up the figure with subplots
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('ML Benchmark Results Analysis', fontsize=14, fontweight='bold')
    
    # --- 1. Duration by Model Type (Box Plot) ---
    ax1 = axes[0, 0]
    models = viz_df['MODEL_CLASS'].unique()
    model_data = [viz_df[viz_df['MODEL_CLASS'] == m]['DURATION_SECONDS'].values for m in models]
    bp = ax1.boxplot(model_data, labels=models, patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set3(np.linspace(0, 1, len(models)))):
        patch.set_facecolor(color)
    ax1.set_ylabel('Duration (seconds)')
    ax1.set_title('Duration by Model Type')
    ax1.tick_params(axis='x', rotation=45)
    ax1.set_yscale('log')  # Log scale for wide range
    
    # --- 2. Duration by Compute Pool (Box Plot) ---
    ax2 = axes[0, 1]
    pools = sorted(viz_df['COMPUTE_POOL'].unique())
    pool_data = [viz_df[viz_df['COMPUTE_POOL'] == p]['DURATION_SECONDS'].values for p in pools]
    bp2 = ax2.boxplot(pool_data, labels=[p.replace('_TEST', '') for p in pools], patch_artist=True)
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    for patch, color in zip(bp2['boxes'], colors[:len(pools)]):
        patch.set_facecolor(color)
    ax2.set_ylabel('Duration (seconds)')
    ax2.set_title('Duration by Compute Pool')
    ax2.tick_params(axis='x', rotation=45)
    
    # --- 3. Duration vs Row Count (Scatter) ---
    ax3 = axes[0, 2]
    for model in viz_df['MODEL_CLASS'].unique():
        subset = viz_df[viz_df['MODEL_CLASS'] == model]
        means = subset.groupby('N_ROWS_SAMPLED')['DURATION_SECONDS'].mean()
        ax3.plot(means.index, means.values, 'o-', label=model, alpha=0.7)
    ax3.set_xlabel('Number of Rows')
    ax3.set_ylabel('Mean Duration (seconds)')
    ax3.set_title('Duration vs Data Size (Rows)')
    ax3.legend(fontsize=7, loc='upper left')
    ax3.ticklabel_format(style='scientific', axis='x', scilimits=(0,0))
    
    # --- 4. Heatmap: Mean Duration (Model × Pool) ---
    ax4 = axes[1, 0]
    heatmap_data = viz_df.pivot_table(
        values='DURATION_SECONDS',
        index='MODEL_CLASS',
        columns='COMPUTE_POOL',
        aggfunc='mean'
    )
    im = ax4.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')
    ax4.set_xticks(range(len(heatmap_data.columns)))
    ax4.set_yticks(range(len(heatmap_data.index)))
    ax4.set_xticklabels([c.replace('_TEST', '') for c in heatmap_data.columns], rotation=45, ha='right')
    ax4.set_yticklabels(heatmap_data.index)
    ax4.set_title('Mean Duration Heatmap (Model × Pool)')
    cbar = plt.colorbar(im, ax=ax4, shrink=0.8)
    cbar.set_label('Seconds')
    
    # --- 5. Duration Distribution (Histogram) ---
    ax5 = axes[1, 1]
    ax5.hist(viz_df['DURATION_SECONDS'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax5.axvline(viz_df['DURATION_SECONDS'].median(), color='red', linestyle='--', label=f"Median: {viz_df['DURATION_SECONDS'].median():.1f}s")
    ax5.axvline(viz_df['DURATION_SECONDS'].mean(), color='orange', linestyle='--', label=f"Mean: {viz_df['DURATION_SECONDS'].mean():.1f}s")
    ax5.set_xlabel('Duration (seconds)')
    ax5.set_ylabel('Frequency')
    ax5.set_title('Duration Distribution')
    ax5.legend()
    
    # --- 6. Duration vs Column Count (Line Plot) ---
    ax6 = axes[1, 2]
    col_means = viz_df.groupby(['N_COLS_SAMPLED', 'MODEL_CLASS'])['DURATION_SECONDS'].mean().unstack()
    col_means.plot(ax=ax6, marker='o', alpha=0.7)
    ax6.set_xlabel('Number of Columns')
    ax6.set_ylabel('Mean Duration (seconds)')
    ax6.set_title('Duration vs Feature Count')
    ax6.legend(fontsize=7, loc='upper left')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ Visualizations generated from {len(viz_df):,} successful benchmark runs")

In [ ]:
# ============================================================================
# VISUALIZATIONS: Model Performance & Coverage Analysis  
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np

print("📊 Generating Model Performance & Coverage Visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('ML Duration Predictor Performance & Grid Coverage', fontsize=14, fontweight='bold')

# --- 1. Actual vs Predicted Duration (Regression) ---
ax1 = axes[0, 0]
if 'y_test_reg' in dir() and 'y_pred_reg' in dir():
    ax1.scatter(y_test_reg, y_pred_reg, alpha=0.5, s=20)
    max_val = max(y_test_reg.max(), y_pred_reg.max())
    ax1.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect Prediction')
    ax1.set_xlabel('Actual Duration (seconds)')
    ax1.set_ylabel('Predicted Duration (seconds)')
    ax1.set_title(f'Actual vs Predicted Duration (R²={r2:.3f})')
    ax1.legend()
else:
    ax1.text(0.5, 0.5, 'Run ML training cell first', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title('Actual vs Predicted Duration')

# --- 2. Residuals Distribution ---
ax2 = axes[0, 1]
if 'y_test_reg' in dir() and 'y_pred_reg' in dir():
    residuals = y_test_reg.values - y_pred_reg
    ax2.hist(residuals, bins=40, edgecolor='black', alpha=0.7, color='coral')
    ax2.axvline(0, color='black', linestyle='--', lw=2)
    ax2.axvline(residuals.mean(), color='blue', linestyle='--', label=f'Mean: {residuals.mean():.2f}s')
    ax2.set_xlabel('Residual (Actual - Predicted) seconds')
    ax2.set_ylabel('Frequency')
    ax2.set_title(f'Prediction Residuals (MAE={mae:.2f}s)')
    ax2.legend()
else:
    ax2.text(0.5, 0.5, 'Run ML training cell first', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Prediction Residuals')

# --- 3. Feature Importance (from regressor) ---
ax3 = axes[1, 0]
if 'regressor' in dir() and hasattr(regressor, 'feature_importances_'):
    feature_names = ['Model Type', 'Compute Pool', 'N Columns', 'N Rows']
    importances = regressor.feature_importances_
    idx = np.argsort(importances)
    ax3.barh(range(len(idx)), importances[idx], color='teal')
    ax3.set_yticks(range(len(idx)))
    ax3.set_yticklabels([feature_names[i] for i in idx])
    ax3.set_xlabel('Feature Importance')
    ax3.set_title('Duration Predictor Feature Importance')
else:
    ax3.text(0.5, 0.5, 'Run ML training cell first', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Feature Importance')

# --- 4. Grid Coverage Heatmap (Tested vs Untested) ---
ax4 = axes[1, 1]
if 'coverage_tested_combos' in dir() and 'gaps_target_grid' in dir():
    # Create coverage matrix: rows × cols showing % tested across all models/pools
    rows_list = sorted(gaps_target_grid['rows'])
    cols_list = sorted(gaps_target_grid['cols'])
    n_models = len(gaps_target_grid['models'])
    n_pools = len(gaps_target_grid['pools'])
    total_per_cell = n_models * n_pools
    
    coverage_matrix = np.zeros((len(rows_list), len(cols_list)))
    for i, n_rows in enumerate(rows_list):
        for j, n_cols in enumerate(cols_list):
            tested_count = sum(
                1 for combo in coverage_tested_combos 
                if combo[2] == n_cols and combo[3] == n_rows
            )
            coverage_matrix[i, j] = (tested_count / total_per_cell) * 100
    
    im = ax4.imshow(coverage_matrix, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
    ax4.set_xticks(range(len(cols_list)))
    ax4.set_yticks(range(len(rows_list)))
    ax4.set_xticklabels(cols_list)
    ax4.set_yticklabels([f'{r:,}' for r in rows_list])
    ax4.set_xlabel('Number of Columns')
    ax4.set_ylabel('Number of Rows')
    ax4.set_title('Grid Coverage % (Rows × Cols)')
    
    # Add percentage text to each cell
    for i in range(len(rows_list)):
        for j in range(len(cols_list)):
            val = coverage_matrix[i, j]
            color = 'white' if val < 50 else 'black'
            ax4.text(j, i, f'{val:.0f}%', ha='center', va='center', color=color, fontsize=10)
    
    cbar = plt.colorbar(im, ax=ax4, shrink=0.8)
    cbar.set_label('% Tested')
else:
    ax4.text(0.5, 0.5, 'Run coverage analysis first', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Grid Coverage')

plt.tight_layout()
plt.show()

print("✅ Performance visualizations complete")